In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

activity = pd.read_csv(
    "../data/processed/site_month_clean.csv",
    low_memory=False
)

acs = pd.read_csv(
    "../data/public/acs_county.csv",
    dtype={"fips": str}
)

print("Activity shape:", activity.shape)
print("ACS shape:", acs.shape)

print(
    "\nActivity years:",
    sorted(activity["_Year"].dropna().unique())
)

print(
    "\nACS years:",
    sorted(acs["year"].dropna().unique())
)

Activity shape: (28154, 65)
ACS shape: (261, 9)

Activity years: [2022, 2023, 2024, 2025, 2026]

ACS years: [2022, 2023, 2024]


In [2]:
event1 = activity[
    (activity["_Year"] >= 2022)
    & (activity["_Year"] <= 2024)
    & activity["County"].notna()
    & (activity["stable_type"] != "Meal Program")
].copy()

print(
    "Event 1 activity rows:",
    len(event1)
)

print(
    "Counties:",
    event1["County"].nunique()
)

print(
    "\nRows by year:"
)

print(
    event1["_Year"]
    .value_counts()
    .sort_index()
)

Event 1 activity rows: 16766
Counties: 86

Rows by year:
_Year
2022    5346
2023    5801
2024    5619
Name: count, dtype: int64


In [3]:
event1["reported_visit"] = (
    event1["HouseholdsReg"] > 0
)

reporting = (
    event1
    .groupby("County", as_index=False)
    .agg(
        report_share=(
            "reported_visit",
            "mean"
        )
    )
)

high_reporting = reporting[
    reporting["report_share"] >= 0.90
].copy()

print(
    "Counties with at least 90% reporting:",
    len(high_reporting)
)

display(
    reporting
    .sort_values("report_share")
    .head(15)
)

Counties with at least 90% reporting: 79


,County,report_share
3,Beltrami,0.668831
25,Grant,0.802817
35,Koochiching,0.829352
0,Aitkin,0.843750
30,Itasca,0.859459
13,Clay,0.868132
18,Dakota,0.893960
52,Nobles,0.906250
42,Mahnomen,0.915493
74,Stevens,0.916667


In [4]:
event1_monthly = (
    event1
    .groupby(
        [
            "County",
            "_Year",
            "_Month"
        ],
        as_index=False
    )
    .agg(
        visits=(
            "HouseholdsReg",
            "sum"
        )
    )
)

event1_monthly = event1_monthly[
    event1_monthly["County"].isin(
        high_reporting["County"]
    )
].copy()

event1_monthly["month_id"] = (
    event1_monthly["_Year"].astype(int).astype(str)
    + "-"
    + event1_monthly["_Month"]
        .astype(int)
        .astype(str)
        .str.zfill(2)
)

print(
    "County-month rows:",
    len(event1_monthly)
)

print(
    "Counties:",
    event1_monthly["County"].nunique()
)

print(
    "Months:",
    event1_monthly["month_id"].nunique()
)

display(
    event1_monthly.head()
)

County-month rows: 2844
Counties: 79
Months: 36


,County,_Year,_Month,visits,month_id
36,Anoka,2022,1,8459.0,2022-01
37,Anoka,2022,2,22209.0,2022-02
38,Anoka,2022,3,13587.0,2022-03
39,Anoka,2022,4,13785.0,2022-04
40,Anoka,2022,5,12414.0,2022-05


In [5]:
snap_2022 = acs[
    acs["year"] == 2022
][
    [
        "county",
        "snap_hh_rate"
    ]
].copy()

snap_2022 = snap_2022.rename(
    columns={
        "county": "County"
    }
)

print(
    "2022 SNAP counties:",
    len(snap_2022)
)

print(
    "Missing SNAP rates:",
    snap_2022["snap_hh_rate"].isna().sum()
)

display(
    snap_2022.head()
)

2022 SNAP counties: 87
Missing SNAP rates: 0


,County,snap_hh_rate
0,Aitkin,0.076504
1,Anoka,0.064216
2,Becker,0.107188
3,Beltrami,0.137269
4,Benton,0.076324


In [6]:
event1_monthly = event1_monthly.merge(
    snap_2022,
    on="County",
    how="left",
    validate="many_to_one"
)

print(
    "Missing SNAP after merge:",
    event1_monthly["snap_hh_rate"].isna().sum()
)

print(
    "Rows after merge:",
    len(event1_monthly)
)

Missing SNAP after merge: 0
Rows after merge: 2844


In [7]:
snap_mean = (
    event1_monthly[
        ["County", "snap_hh_rate"]
    ]
    .drop_duplicates()["snap_hh_rate"]
    .mean()
)

snap_sd = (
    event1_monthly[
        ["County", "snap_hh_rate"]
    ]
    .drop_duplicates()["snap_hh_rate"]
    .std()
)

event1_monthly["snap_z"] = (
    event1_monthly["snap_hh_rate"]
    - snap_mean
) / snap_sd

print(
    "SNAP mean:",
    snap_mean
)

print(
    "SNAP SD:",
    snap_sd
)

print(
    "\nStandardized county exposure:"
)

print(
    event1_monthly[
        ["County", "snap_z"]
    ]
    .drop_duplicates()["snap_z"]
    .describe()
)

SNAP mean: 0.07626598236375397
SNAP SD: 0.028212401347959232

Standardized county exposure:
count    7.900000e+01
mean     2.332874e-16
std      1.000000e+00
min     -1.502611e+00
25%     -5.971800e-01
50%     -1.993915e-01
75%      3.866398e-01
max      5.181862e+00
Name: snap_z, dtype: float64


In [8]:
event1_monthly["date"] = pd.to_datetime(
    dict(
        year=event1_monthly["_Year"].astype(int),
        month=event1_monthly["_Month"].astype(int),
        day=1
    )
)

event1_monthly["post_apr2023"] = (
    event1_monthly["date"]
    >= pd.Timestamp("2023-04-01")
).astype(int)

print(
    event1_monthly[
        [
            "date",
            "post_apr2023"
        ]
    ]
    .drop_duplicates()
    .sort_values("date")
    .tail(18)
)

         date  post_apr2023
18 2023-07-01             1
19 2023-08-01             1
20 2023-09-01             1
21 2023-10-01             1
22 2023-11-01             1
23 2023-12-01             1
24 2024-01-01             1
25 2024-02-01             1
26 2024-03-01             1
27 2024-04-01             1
28 2024-05-01             1
29 2024-06-01             1
30 2024-07-01             1
31 2024-08-01             1
32 2024-09-01             1
33 2024-10-01             1
34 2024-11-01             1
35 2024-12-01             1


In [9]:
event1_model = smf.glm(
    formula=(
        "visits ~ "
        "C(County) + "
        "C(month_id) + "
        "snap_z:post_apr2023"
    ),
    data=event1_monthly,
    family=sm.families.Poisson()
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": event1_monthly["County"]
    }
)

print(
    event1_model.summary()
)

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 visits   No. Observations:                 2844
Model:                            GLM   Df Residuals:                     2729
Model Family:                 Poisson   Df Model:                          114
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0296e+05
Date:                Wed, 14 Oct 2026   Deviance:                   1.8303e+05
Time:                        15:48:52   Pearson chi2:                 1.96e+05
No. Iterations:                     8   Pseudo R-squ. (CS):              1.000
Covariance Type:              cluster                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [10]:
theta = event1_model.params[
    "snap_z:post_apr2023"
]

theta_se = event1_model.bse[
    "snap_z:post_apr2023"
]

theta_p = event1_model.pvalues[
    "snap_z:post_apr2023"
]

percent_change = (
    np.exp(theta) - 1
) * 100

print(
    "Interaction coefficient:",
    theta
)

print(
    "Clustered SE:",
    theta_se
)

print(
    "p-value:",
    theta_p
)

print(
    "Percent change per 1-SD higher SNAP rate:",
    percent_change
)

Interaction coefficient: -0.19185467387020316
Clustered SE: 0.08351423539821978
p-value: 0.021603422366578844
Percent change per 1-SD higher SNAP rate: -17.45731841396927


In [11]:
# Hennepin, Ramsey, Washington, and Steele were influential. Treating this strictly as a sensitivity analysis, not as a replacement for the full-sample result.
influential_counties = [
    "Hennepin",
    "Ramsey",
    "Washington",
    "Steele"
]

event1_restricted = event1_monthly[
    ~event1_monthly["County"].isin(
        influential_counties
    )
].copy()

print(
    "Counties in full sample:",
    event1_monthly["County"].nunique()
)

print(
    "Counties after removal:",
    event1_restricted["County"].nunique()
)

print(
    "Rows after removal:",
    len(event1_restricted)
)

event1_model_restricted = smf.glm(
    formula=(
        "visits ~ "
        "C(County) + "
        "C(month_id) + "
        "snap_z:post_apr2023"
    ),
    data=event1_restricted,
    family=sm.families.Poisson()
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": event1_restricted["County"]
    }
)

Counties in full sample: 79
Counties after removal: 75
Rows after removal: 2700


In [12]:
theta_restricted = event1_model_restricted.params[
    "snap_z:post_apr2023"
]

se_restricted = event1_model_restricted.bse[
    "snap_z:post_apr2023"
]

p_restricted = event1_model_restricted.pvalues[
    "snap_z:post_apr2023"
]

pct_restricted = (
    np.exp(theta_restricted) - 1
) * 100

comparison = pd.DataFrame(
    {
        "Model": [
            "Full sample",
            "Remove 4 influential counties"
        ],
        "Coefficient": [
            theta,
            theta_restricted
        ],
        "Clustered_SE": [
            theta_se,
            se_restricted
        ],
        "p_value": [
            theta_p,
            p_restricted
        ],
        "Percent_change": [
            percent_change,
            pct_restricted
        ]
    }
)

display(comparison)

,Model,Coefficient,Clustered_SE,p_value,Percent_change
0,Full sample,-0.191855,0.083514,0.021603,-17.457318
1,Remove 4 influential counties,-0.071230,0.036135,0.048697,-6.875207


In [13]:
placebo = event1_monthly[
    event1_monthly["_Year"] == 2022
].copy()

placebo["post_jul2022"] = (
    placebo["date"]
    >= pd.Timestamp("2022-07-01")
).astype(int)

print(
    "Placebo counties:",
    placebo["County"].nunique()
)

print(
    "Placebo months:",
    placebo["month_id"].nunique()
)

print(
    "Placebo rows:",
    len(placebo)
)

Placebo counties: 79
Placebo months: 12
Placebo rows: 948


In [14]:
placebo_model = smf.glm(
    formula=(
        "visits ~ "
        "C(County) + "
        "C(month_id) + "
        "snap_z:post_jul2022"
    ),
    data=placebo,
    family=sm.families.Poisson()
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": placebo["County"]
    }
)

placebo_theta = placebo_model.params[
    "snap_z:post_jul2022"
]

placebo_se = placebo_model.bse[
    "snap_z:post_jul2022"
]

placebo_p = placebo_model.pvalues[
    "snap_z:post_jul2022"
]

placebo_pct = (
    np.exp(placebo_theta) - 1
) * 100

print(
    "Placebo coefficient:",
    placebo_theta
)

print(
    "Clustered SE:",
    placebo_se
)

print(
    "p-value:",
    placebo_p
)

print(
    "Percent change:",
    placebo_pct
)

Placebo coefficient: 0.008067098994409327
Clustered SE: 0.02357913982069903
p-value: 0.7322540863843465
Percent change: 0.8099725712812811


In [15]:
cutoff_dates = [
    "2023-03-01",
    "2023-04-01",
    "2023-05-01"
]

cutoff_results = []

for cutoff in cutoff_dates:

    test_data = event1_monthly.copy()

    test_data["post_test"] = (
        test_data["date"]
        >= pd.Timestamp(cutoff)
    ).astype(int)

    test_model = smf.glm(
        formula=(
            "visits ~ "
            "C(County) + "
            "C(month_id) + "
            "snap_z:post_test"
        ),
        data=test_data,
        family=sm.families.Poisson()
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": test_data["County"]
        }
    )

    coefficient = test_model.params[
        "snap_z:post_test"
    ]

    standard_error = test_model.bse[
        "snap_z:post_test"
    ]

    p_value = test_model.pvalues[
        "snap_z:post_test"
    ]

    percent_change_test = (
        np.exp(coefficient) - 1
    ) * 100

    cutoff_results.append(
        {
            "cutoff": cutoff,
            "coefficient": coefficient,
            "clustered_se": standard_error,
            "p_value": p_value,
            "percent_change": percent_change_test
        }
    )

cutoff_table = pd.DataFrame(
    cutoff_results
)

display(cutoff_table)

,cutoff,coefficient,clustered_se,p_value,percent_change
0,2023-03-01,-0.205070,0.085364,0.016292,-18.540986
1,2023-04-01,-0.191855,0.083514,0.021603,-17.457318
2,2023-05-01,-0.182837,0.079815,0.021977,-16.709595


In [16]:
restricted_cutoff_results = []

for cutoff in cutoff_dates:

    test_data = event1_restricted.copy()

    test_data["post_test"] = (
        test_data["date"]
        >= pd.Timestamp(cutoff)
    ).astype(int)

    test_model = smf.glm(
        formula=(
            "visits ~ "
            "C(County) + "
            "C(month_id) + "
            "snap_z:post_test"
        ),
        data=test_data,
        family=sm.families.Poisson()
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": test_data["County"]
        }
    )

    coefficient = test_model.params[
        "snap_z:post_test"
    ]

    standard_error = test_model.bse[
        "snap_z:post_test"
    ]

    p_value = test_model.pvalues[
        "snap_z:post_test"
    ]

    percent_change_test = (
        np.exp(coefficient) - 1
    ) * 100

    restricted_cutoff_results.append(
        {
            "cutoff": cutoff,
            "coefficient": coefficient,
            "clustered_se": standard_error,
            "p_value": p_value,
            "percent_change": percent_change_test
        }
    )

restricted_cutoff_table = pd.DataFrame(
    restricted_cutoff_results
)

display(restricted_cutoff_table)

,cutoff,coefficient,clustered_se,p_value,percent_change
0,2023-03-01,-0.071531,0.035733,0.045307,-6.903241
1,2023-04-01,-0.071230,0.036135,0.048697,-6.875207
2,2023-05-01,-0.073105,0.036912,0.047645,-7.049680


In [17]:
event2 = activity[
    (activity["_Year"] >= 2025)
    & (activity["_Year"] <= 2026)
    & activity["County"].notna()
    & (activity["stable_type"] != "Meal Program")
].copy()

print(
    "Event 2 rows:",
    len(event2)
)

print(
    "\nRows by year:"
)

print(
    event2["_Year"]
    .value_counts()
    .sort_index()
)

print(
    "\nMonths available by year:"
)

print(
    event2.groupby("_Year")["_Month"]
    .unique()
)

Event 2 rows: 8667

Rows by year:
_Year
2025    5475
2026    3192
Name: count, dtype: int64

Months available by year:
_Year
2025    [11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
2026                      [1, 2, 3, 4, 5, 6, 7]
Name: _Month, dtype: object


In [18]:
event2["reported_visit"] = (
    event2["HouseholdsReg"] > 0
)

reporting_event2 = (
    event2
    .groupby(
        "County",
        as_index=False
    )
    .agg(
        report_share=(
            "reported_visit",
            "mean"
        )
    )
)

high_reporting_event2 = reporting_event2[
    reporting_event2["report_share"] >= 0.90
].copy()

print(
    "Counties with at least 90% reporting:",
    len(high_reporting_event2)
)

display(
    reporting_event2
    .sort_values("report_share")
    .head(15)
)

Counties with at least 90% reporting: 84


,County,report_share
13,Clay,0.820000
3,Beltrami,0.834783
47,Mille Lacs,0.915789
26,Hennepin,0.921617
30,Itasca,0.937500
61,Ramsey,0.938462
42,Mahnomen,0.947368
25,Grant,0.947368
0,Aitkin,0.951456
1,Anoka,0.953488


In [19]:
event2_monthly = (
    event2
    .groupby(
        [
            "County",
            "_Year",
            "_Month"
        ],
        as_index=False
    )
    .agg(
        visits=(
            "HouseholdsReg",
            "sum"
        )
    )
)

event2_monthly = event2_monthly[
    event2_monthly["County"].isin(
        high_reporting_event2["County"]
    )
].copy()

event2_monthly["date"] = pd.to_datetime(
    dict(
        year=event2_monthly["_Year"].astype(int),
        month=event2_monthly["_Month"].astype(int),
        day=1
    )
)

event2_monthly["month_id"] = (
    event2_monthly["_Year"]
    .astype(int)
    .astype(str)
    + "-"
    + event2_monthly["_Month"]
      .astype(int)
      .astype(str)
      .str.zfill(2)
)

print(
    "County-month rows:",
    len(event2_monthly)
)

print(
    "Counties:",
    event2_monthly["County"].nunique()
)

print(
    "Months:",
    event2_monthly["month_id"].nunique()
)

print(
    "First month:",
    event2_monthly["date"].min()
)

print(
    "Last month:",
    event2_monthly["date"].max()
)

County-month rows: 1596
Counties: 84
Months: 19
First month: 2025-01-01 00:00:00
Last month: 2026-07-01 00:00:00


In [20]:
snap_2024 = acs[
    acs["year"] == 2024
][
    [
        "county",
        "snap_hh_rate"
    ]
].copy()

snap_2024 = snap_2024.rename(
    columns={
        "county": "County"
    }
)

event2_monthly = event2_monthly.merge(
    snap_2024,
    on="County",
    how="left",
    validate="many_to_one"
)

print(
    "Missing SNAP rates:",
    event2_monthly[
        "snap_hh_rate"
    ].isna().sum()
)

Missing SNAP rates: 0


In [21]:
event2_snap_county = (
    event2_monthly[
        [
            "County",
            "snap_hh_rate"
        ]
    ]
    .drop_duplicates()
)

event2_snap_mean = (
    event2_snap_county[
        "snap_hh_rate"
    ].mean()
)

event2_snap_sd = (
    event2_snap_county[
        "snap_hh_rate"
    ].std()
)

event2_monthly["snap_z"] = (
    event2_monthly["snap_hh_rate"]
    - event2_snap_mean
) / event2_snap_sd

print(
    "SNAP mean:",
    event2_snap_mean
)

print(
    "SNAP SD:",
    event2_snap_sd
)

SNAP mean: 0.08408325212945243
SNAP SD: 0.029977434802302288


In [22]:
yoy_data = activity[
    (
        activity["_Year"].isin(
            [2024, 2025]
        )
    )
    & (
        activity["_Month"].isin(
            [10, 11, 12]
        )
    )
    & activity["County"].notna()
    & (
        activity["stable_type"]
        != "Meal Program"
    )
].copy()

yoy_monthly = (
    yoy_data
    .groupby(
        [
            "County",
            "_Year",
            "_Month"
        ],
        as_index=False
    )
    .agg(
        visits=(
            "HouseholdsReg",
            "sum"
        )
    )
)

# Use the same high-reporting counties
# selected for Event 2.
yoy_monthly = yoy_monthly[
    yoy_monthly["County"].isin(
        high_reporting_event2["County"]
    )
].copy()

print(
    "Rows:",
    len(yoy_monthly)
)

print(
    "Counties:",
    yoy_monthly["County"].nunique()
)

print(
    "Observations per county:"
)

print(
    yoy_monthly.groupby(
        "County"
    ).size().value_counts()
)

Rows: 504
Counties: 84
Observations per county:
6    84
Name: count, dtype: int64


In [23]:
yoy_monthly = yoy_monthly.merge(
    snap_2024,
    on="County",
    how="left",
    validate="many_to_one"
)

snap_county_yoy = (
    yoy_monthly[
        [
            "County",
            "snap_hh_rate"
        ]
    ]
    .drop_duplicates()
)

yoy_snap_mean = (
    snap_county_yoy[
        "snap_hh_rate"
    ].mean()
)

yoy_snap_sd = (
    snap_county_yoy[
        "snap_hh_rate"
    ].std()
)

yoy_monthly["snap_z"] = (
    yoy_monthly["snap_hh_rate"]
    - yoy_snap_mean
) / yoy_snap_sd

yoy_monthly["year2025"] = (
    yoy_monthly["_Year"] == 2025
).astype(int)

print(
    "Missing SNAP:",
    yoy_monthly[
        "snap_hh_rate"
    ].isna().sum()
)

print(
    "SNAP mean:",
    yoy_snap_mean
)

print(
    "SNAP SD:",
    yoy_snap_sd
)

Missing SNAP: 0
SNAP mean: 0.08408325212945243
SNAP SD: 0.029977434802302288


In [24]:
month_results = []

month_numbers = [
    10,
    11,
    12
]

month_names = [
    "October",
    "November",
    "December"
]

for month_number, month_name in zip(
    month_numbers,
    month_names
):

    month_data = yoy_monthly[
        yoy_monthly["_Month"]
        == month_number
    ].copy()

    model = smf.glm(
        formula=(
            "visits ~ "
            "C(County) + "
            "C(_Year) + "
            "snap_z:year2025"
        ),
        data=month_data,
        family=sm.families.Poisson()
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                month_data["County"]
        }
    )

    coefficient = model.params[
        "snap_z:year2025"
    ]

    standard_error = model.bse[
        "snap_z:year2025"
    ]

    p_value = model.pvalues[
        "snap_z:year2025"
    ]

    percent_change = (
        np.exp(coefficient) - 1
    ) * 100

    month_results.append(
        {
            "Month": month_name,
            "Coefficient": coefficient,
            "Clustered_SE":
                standard_error,
            "p_value": p_value,
            "Percent_change":
                percent_change
        }
    )

yoy_results = pd.DataFrame(
    month_results
)

display(yoy_results)

,Month,Coefficient,Clustered_SE,p_value,Percent_change
0,October,-0.006160,0.066039,0.925678,-0.614143
1,November,0.036755,0.067189,0.584349,3.743918
2,December,-0.057128,0.061039,0.349315,-5.552661


Overall SNAP-shock evidence. The April 2023 event produces a negative association: approximately -17.5% per one-standard-deviation higher SNAP exposure in the full sample (\(p=0.022\)) and -6.9% after removing influential counties (\(p=0.049\)). The estimate remains negative across March, April, and May cutoffs, while the July 2022 placebo is null. Event 2 is also null. Therefore, the prediction that higher-SNAP counties would show larger increases in food-shelf use when SNAP support fell is not supported by these descriptive tests. These results are associations and should not be interpreted causally.

Overall SNAP-shock evidence. The April 2023 event produces a negative association that weakens after influential counties are removed, while the November 2025 event produces a positive but statistically imprecise association. Because the two events point in different directions and neither result is robust enough for causal interpretation, these tests neither confirm nor reject the SNAP component of the need framework.